<a href="https://colab.research.google.com/github/Davidson-Open-University/csc203-labs/blob/main/notebooks/CSC203-Lab3-Truth-Tables-in-Python.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSC 203 Lab 3: Truth Tables in Python

**Study Session 3: Logic** · Davidson Open University Global

**In this lab you will:**
- build truth tables automatically for statements with 2, 3 or more variables;
- test tautologies and logical equivalence (for example De Morgan's laws);
- decide whether a rule of inference is valid;
- evaluate statements with quantifiers ∀ and ∃ over a finite domain.

**Time:** about 60 minutes in four parts:
- **Part A – Explore** (about 10 minutes): run the examples and read the explanations.
- **Part B – Practise** (about 20 minutes): fill in the blanks. A self-check tells you if you are right.
- **Part C – Build a tool** (about 20 minutes): complete a short Python function and test it.
- **Part D – Apply and reflect** (about 10 minutes): use your tools on your own example and write short answers.

**How to use this notebook**
1. Run each code cell in order: click it and press **Shift + Enter** (or the ▶ button).
2. Where you see `...`, replace it with your own code, run the cell, then run the **check** cell under it.
   - ✅ means correct. ❌ gives you a hint: try again. ⏳ means you have not filled in the blank yet.
   - The checks never show the answer, so you learn by working it out. You can try as many times as you like.
3. Stuck? Read the hint, look back at the study session, or ask in the discussion forum. Do not skip ahead: each part builds on the last.
4. Save your work: in Colab choose **File → Save a copy in GitHub** and save it to your own `csc203-labs` repository. Offline in Jupyter, press **Ctrl + S** and later upload the file with **Add file → Upload files** on GitHub.

*This lab is practice to help you learn and prepare for the assignment. It is not graded unless your tutor tells you otherwise.*

In [ ]:
#@title ▶ Run this cell first: it sets up the self-checks (you do not need to read it)
# You do not need to read or change this code.
import hashlib as _h
class _Blank(Exception): pass
def _norm(x, unordered=False):
    try:
        import numpy as _np
        if isinstance(x, _np.ndarray): x = x.tolist()
        elif isinstance(x, _np.bool_): x = bool(x)
        elif isinstance(x, _np.integer): x = int(x)
        elif isinstance(x, _np.floating): x = float(x)
    except ImportError:
        pass
    if x is Ellipsis or x is None: raise _Blank()
    if isinstance(x, bool): return 'b:' + str(x)
    if isinstance(x, (int, float)): return 'n:' + repr(round(float(x), 6) + 0.0)
    if isinstance(x, str): return 't:' + ' '.join(x.lower().split())
    if isinstance(x, (set, frozenset)): return 's{' + ','.join(sorted(_norm(i) for i in x)) + '}'
    if isinstance(x, (list, tuple)):
        items = [_norm(i) for i in x]
        if unordered: items.sort()
        return 'l[' + ','.join(items) + ']'
    if isinstance(x, dict): return 'd{' + ','.join(sorted(_norm(k) + '=' + _norm(v) for k, v in x.items())) + '}'
    try:
        import sympy as _sp
        if isinstance(x, _sp.Basic):
            x = _sp.sympify(x)
            if x.is_number: return 'n:' + repr(round(float(x), 6) + 0.0)
            syms = sorted(x.free_symbols, key=str)
            vals = []
            for t in (0.3, 0.7, 1.3):
                v = complex(x.subs({s: t + 0.11 * i for i, s in enumerate(syms)}).evalf())
                vals.append(round(v.real, 6) + 0.0)
            return 'e:' + repr(vals)
    except ImportError:
        pass
    return 'r:' + repr(x)
def _hash(q, s): return _h.sha256(('csc203|' + q + '|' + s).encode()).hexdigest()[:16]
def check(q, value):
    """Check your answer to question q. Prints ✅, ❌ with a hint, or ⏳ if the blank is not filled yet."""
    spec = _CHECKS[q]
    try:
        if callable(value) and 'args' in spec:
            outs = []
            for a in spec['args']:
                try:
                    outs.append(value(*a))
                except _Blank:
                    raise
                except Exception as e:
                    print(f'❌ Question {q}: your function stopped with an error: {type(e).__name__}: {e}')
                    return
            if all(o is None or o is Ellipsis for o in outs): raise _Blank()
            got = _norm(outs, False)
        elif spec.get('logic'):
            if value is None or value is Ellipsis: raise _Blank()
            import sympy as _sp
            from itertools import product as _p
            syms = sorted(_sp.sympify(value).free_symbols, key=str)
            got = 'L' + ''.join('1' if bool(_sp.sympify(value).subs(dict(zip(syms, v)))) else '0' for v in _p([True, False], repeat=len(syms))) + ':' + ','.join(map(str, syms))
        else:
            got = _norm(value, spec.get('unordered', False))
    except _Blank:
        print(f'⏳ Question {q}: fill in the blank (replace ...) and run the cell above, then run this cell again.')
        return
    if _RECORD is not None:
        _RECORD[q] = _hash(q, got); print(f'✅ Question {q}: recorded.'); return
    if _hash(q, got) == spec['h']:
        print(f'✅ Question {q}: correct! ' + spec.get('ok', 'Well done.'))
    else:
        print(f'❌ Question {q}: not yet. Hint: ' + spec['hint'])
_RECORD = None

_CHECKS = {
    '3.1': {'h': '3107a00851298014', 'hint': 'Each variable doubles the number of rows: 2 × 2 × 2 × 2.', 'ok': 'In general, n variables give 2ⁿ rows.'},
    '3.2': {'h': '103a97461288fbbc', 'hint': 'Use implies(...): the first input is (P and Q), the second is R.', 'ok': 'The statement is only false when P and Q are true and R is false.'},
    '3.3': {'h': '019af54cf24a0d5d', 'hint': 'Test all four letters. The converse is NOT equivalent to the original statement, but the contrapositive is.', 'ok': 'A statement is always equivalent to its contrapositive, but not to its converse. You will use this in Assignment 3.'},
    '3.4': {'h': 'fceba0b859d154ed', 'hint': 'Return all(statement(*row) for row in product([True, False], repeat=n)).', 'ok': 'Your tautology tester works.', 'args': [(1, lambda P: P or not P), (1, lambda P: P), (2, lambda P, Q: (not P) or Q or P), (2, lambda P, Q: P and Q), (3, lambda P, Q, R: ((not P) or Q) or ((not Q) or R))]},
    '3.5': {'h': '442f9e97cb33bc5f', 'hint': 'Copy the modus ponens pattern: implies(premise1 and premise2, conclusion). Then keep the names where is_tautology(n, s) is True.', 'ok': 'Two of the forms are common fallacies: they look reasonable but are invalid.'},
    '3.6': {'h': '66ecc8c6a2cc91cf', 'hint': 'For (c), think about x = 10: is there a larger y in D? The order of the quantifiers matters.', 'ok': 'Statement (c) would be true on the infinite domain ℕ, but it is false on a finite domain.'},
}
print('Self-checks ready. Now carry on with Part A.')

## Part A – Explore (about 10 minutes)
In Python, `True` and `False` are truth values, and the connectives are:

| Logic | Python |
|---|---|
| ¬P | `not P` |
| P ∧ Q | `P and Q` |
| P ∨ Q | `P or Q` |
| P → Q | `(not P) or Q` |
| P ↔ Q | `P == Q` |

Python has no arrow symbol, so we define a function `implies(P, Q)`. `product([True, False], repeat=2)` lists every row of a truth table for 2 variables.

In [ ]:
from itertools import product

def implies(P, Q):
    return (not P) or Q

def truth_table(names, statement):
    """Print the truth table of `statement`, a function of len(names) variables."""
    print(' '.join(f'{n:^5}' for n in names), '| result')
    for row in product([True, False], repeat=len(names)):
        value = statement(*row)
        print(' '.join(f'{"T" if v else "F":^5}' for v in row), '|  ', 'T' if value else 'F')

truth_table(['P', 'Q'], implies)

**Predict:** P → Q is false in only one row. Which one? Check your answer in the output above.

A **lambda** is a short, one-line function. `lambda P, Q: P and not Q` means "the function that takes P and Q and returns P ∧ ¬Q". Run the truth table of (P ∧ Q) ∨ ¬R for three variables:

In [ ]:
truth_table(['P', 'Q', 'R'], lambda P, Q, R: (P and Q) or not R)

## Part B – Practise (about 20 minutes)
**B1.** How many rows does a truth table with **4** variables have? Let Python count the rows that `product` makes.

In [ ]:
answer_3_1 = ...   # hint: len(list(product([True, False], repeat=4)))

In [ ]:
check('3.1', answer_3_1)

**B2.** Write the statement (P ∧ Q) → R as a lambda. The cell then collects its result column, row by row, in the standard order (TTT, TTF, TFT, …).

In [ ]:
statement_3_2 = ...      # YOUR CODE: lambda P, Q, R: ...
answer_3_2 = [statement_3_2(*row) for row in product([True, False], repeat=3)] if statement_3_2 is not ... else ...
print(answer_3_2)

In [ ]:
check('3.2', answer_3_2)

Two statements are **logically equivalent** (≡) if they have the same value in every row. Here is a function that tests this, followed by four statements. Run it.

In [ ]:
def equivalent(n, s1, s2):
    return all(s1(*row) == s2(*row) for row in product([True, False], repeat=n))

candidates = {
    'a': lambda P, Q: (not P) or Q,          # ¬P ∨ Q
    'b': lambda P, Q: implies(Q, P),         # Q → P   (the converse)
    'c': lambda P, Q: implies(not Q, not P), # ¬Q → ¬P (the contrapositive)
    'd': lambda P, Q: not (P and not Q),     # ¬(P ∧ ¬Q)
}
print("De Morgan: ¬(P ∧ Q) ≡ ¬P ∨ ¬Q ?", equivalent(2, lambda P, Q: not (P and Q), lambda P, Q: (not P) or (not Q)))

**B3.** Which of the candidates a, b, c, d are equivalent to P → Q? Use `equivalent(2, candidates[x], implies)` for each letter, and make `answer_3_3` the **set of letters** that are equivalent, for example `{'a', 'b'}`.

In [ ]:
answer_3_3 = ...

In [ ]:
check('3.3', answer_3_3)

## Part C – Build a tool (about 20 minutes)
### C1. Tautology tester
A **tautology** is true in every row. Complete the function: it should return `True` if `statement` (a function of n variables) is true for every row of `product([True, False], repeat=n)`, and `False` otherwise. *Hint:* `all(...)`, as in `equivalent`.

In [ ]:
def is_tautology(n, statement):
    # YOUR CODE
    ...

print(is_tautology(1, lambda P: P or not P))

In [ ]:
check('3.4', is_tautology)

### C2. Valid and invalid arguments
An argument with premises A₁, …, Aₖ and conclusion C is **valid** when (A₁ ∧ … ∧ Aₖ) → C is a tautology. Use your `is_tautology` to test the five argument forms below, and make `answer_3_5` the **set of names** of the valid ones.

| Name | Premises | Conclusion |
|---|---|---|
| `'modus ponens'` | P → Q, P | Q |
| `'modus tollens'` | P → Q, ¬Q | ¬P |
| `'affirming the consequent'` | P → Q, Q | P |
| `'denying the antecedent'` | P → Q, ¬P | ¬Q |
| `'hypothetical syllogism'` | P → Q, Q → R | P → R |

In [ ]:
forms = {
    'modus ponens':             (2, lambda P, Q: implies(implies(P, Q) and P, Q)),
    'modus tollens':            (2, lambda P, Q: ...),   # YOUR CODE
    'affirming the consequent': (2, lambda P, Q: ...),   # YOUR CODE
    'denying the antecedent':   (2, lambda P, Q: ...),   # YOUR CODE
    'hypothetical syllogism':   (3, lambda P, Q, R: ...),# YOUR CODE
}
answer_3_5 = ...   # the set of names of the valid forms

In [ ]:
check('3.5', answer_3_5)

### C3. Quantifiers on a finite domain
For a finite domain D, Python's `all` is the universal quantifier ∀ and `any` is the existential quantifier ∃. For example, ∀x ∈ D (x > 0) is `all(x > 0 for x in D)`.

With D = {1, 2, …, 10}, find the truth value of each statement and store the three answers as a tuple `(a, b, c)`:
- (a) ∀x ∈ D: x² ≥ x
- (b) ∃x ∈ D: x² = 50
- (c) ∀x ∈ D ∃y ∈ D: y > x

In [ ]:
D = range(1, 11)
a = ...
b = ...
c = ...      # hint: all(any(y > x for y in D) for x in D)
answer_3_6 = (a, b, c)
print(answer_3_6)

In [ ]:
check('3.6', answer_3_6 if ... not in answer_3_6 else ...)

## Part D – Apply and reflect (about 10 minutes)
**D1. Prepare for Assignment 3.** Choose an everyday "if P then Q" argument (a school rule or an advert). In the cell, print its truth table with `truth_table`, and use `equivalent` to compare it with its converse and its contrapositive.

In [ ]:
# D1: your own argument
# P = "..."
# Q = "..."

**D2. Reflect.** Answer in one or two sentences each:
1. Explain, using an everyday example, why "affirming the consequent" is a fallacy.
2. Why can a computer check a tautology with 3 variables easily, but struggle with 60 variables?
3. Why does the order of ∀ and ∃ matter in statement (c)?

*Type your answer here (double-click this cell to edit it).*

## Finished? Save your work
In Colab: **File → Save a copy in GitHub** → your `csc203-labs` repository → commit message *"Lab 3 completed"*.
Offline: save with **Ctrl + S**, then upload the file to your repository with **Add file → Upload files** when you are online.

Then go back to Moodle and mark the lab as done.